# Fine-tuning Mistral 7B for Medical Dialogue

This notebook performs fine-tuning of the Mistral 7B Instruct model on medical dialogue data using QLoRA (Quantized Low-Rank Adaptation). The process includes:

1. Loading preprocessed medical dialogue data
2. Setting up the Mistral 7B model with QLoRA configuration
3. Training the model with efficient memory handling
4. Evaluating the model's performance
5. Testing with sample medical queries

## Import Libraries and Set Up Environment

In [1]:
import os
import json
import torch
import logging
import numpy as np
from pathlib import Path
from datasets import Dataset
import bitsandbytes as bnb
from peft import (
    LoraConfig, 
    prepare_model_for_kbit_training,
    get_peft_model, 
    PeftModel,
    TaskType
)
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    HfArgumentParser, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
from transformers.trainer_utils import get_last_checkpoint

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## Configure Training Settings

In [2]:
# Configuration
class Config:
    # Data paths
    PROCESSED_DATA_DIR = Path("processed_data")
    TRAIN_PATH = PROCESSED_DATA_DIR / "processed_train.json"
    DEV_PATH = PROCESSED_DATA_DIR / "processed_dev.json"
    
    # Model configuration
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
    OUTPUT_DIR = Path("model_output")
    
    # Training parameters
    LORA_RANK = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    
    NUM_EPOCHS = 5
    LEARNING_RATE = 2e-4
    BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8
    MAX_SEQUENCE_LENGTH = 256
    
    # Evaluation and saving parameters
    EVAL_STRATEGY = "steps"  # Renamed to match the parameter name in TrainingArguments
    EVAL_STEPS = 500
    SAVE_STRATEGY = "steps"
    SAVE_STEPS = 500
    SAVE_TOTAL_LIMIT = 3
    
    # Directories
    def __init__(self):
        # Create output directory if it doesn't exist
        self.OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

config = Config()

## Load Preprocessed Datasets

In [4]:
def load_json_data(file_path):
    """
    Load JSON data from the specified file path with error handling.
    
    Args:
        file_path (Path): Path to the JSON file
        
    Returns:
        list: List of dialogue data or empty list if file not found
    """
    try:
        if not file_path.exists():
            logger.warning(f"File not found: {file_path}")
            return []
        
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        logger.info(f"Successfully loaded {len(data)} dialogues from {file_path}")
        return data
    except json.JSONDecodeError:
        logger.error(f"Error decoding JSON from {file_path}")
        return []
    except Exception as e:
        logger.error(f"Error loading data from {file_path}: {str(e)}")
        return []

# Load train and dev datasets
train_data = load_json_data(config.TRAIN_PATH)
dev_data = load_json_data(config.DEV_PATH)

# Check if data was loaded successfully
if not train_data:
    raise ValueError(f"No training data found at {config.TRAIN_PATH}. Run data_preprocessing.ipynb first.")
    
# Convert to datasets
train_dataset = Dataset.from_dict({"text": [item["text"] for item in train_data]})
dev_dataset = Dataset.from_dict({"text": [item["text"] for item in dev_data]}) if dev_data else None

print(f"Training examples: {len(train_dataset)}")
if dev_dataset:
    print(f"Development examples: {len(dev_dataset)}")

2025-08-28 18:16:19,835 - INFO - Successfully loaded 482 dialogues from processed_data/processed_train.json


2025-08-28 18:16:19,836 - INFO - Successfully loaded 60 dialogues from processed_data/processed_dev.json


Training examples: 482
Development examples: 60


## Load Model and Tokenizer

In [5]:
def load_model_and_tokenizer():
    """
    Load the Mistral 7B model and tokenizer with QLoRA configuration.
    
    Returns:
        tuple: (model, tokenizer)
    """
    logger.info(f"Loading model: {config.MODEL_NAME}")
    
    # Set device mapping for mixed device training (if needed)
    device_map = "auto"
    
    # Configure 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    )
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"  # Set padding on the right side
    
    # Load base model with quantization configuration
    model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_NAME,
        quantization_config=bnb_config,
        device_map=device_map,
        trust_remote_code=True,
        use_cache=False
    )
    
    # Prepare model for k-bit training
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    
    # Configure LoRA
    lora_target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ]
    
    peft_config = LoraConfig(
        r=config.LORA_RANK,
        lora_alpha=config.LORA_ALPHA,
        target_modules=lora_target_modules,
        lora_dropout=config.LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    
    # Get PEFT model
    model = get_peft_model(model, peft_config)
    
    # Print trainable parameters
    trainable_params = 0
    all_params = 0
    
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
            
    logger.info(f"Trainable params: {trainable_params:,d} ({100 * trainable_params / all_params:.2f}%)")
    logger.info(f"All params: {all_params:,d}")
    
    return model, tokenizer

# Load model and tokenizer
model, tokenizer = load_model_and_tokenizer()

2025-08-28 18:16:19,855 - INFO - Loading model: mistralai/Mistral-7B-Instruct-v0.3
2025-08-28 18:16:21,780 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

2025-08-28 18:16:33,052 - INFO - Trainable params: 41,943,040 (1.10%)
2025-08-28 18:16:33,053 - INFO - All params: 3,800,305,664


## Prepare Datasets for Training

In [6]:
def tokenize_function(examples):
    """
    Tokenize the input texts.
    
    Args:
        examples (dict): Dictionary of examples
        
    Returns:
        dict: Dictionary of tokenized examples
    """
    # Tokenize with padding and truncation
    tokenized_inputs = tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=config.MAX_SEQUENCE_LENGTH,
        return_tensors=None
    )
    
    return tokenized_inputs

logger.info("Tokenizing datasets...")
tokenized_train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=["text"]
)

tokenized_dev_dataset = None
if dev_dataset:
    tokenized_dev_dataset = dev_dataset.map(
        tokenize_function,
        batched=True,
        num_proc=4,
        remove_columns=["text"]
    )

print(f"Tokenized training examples: {len(tokenized_train_dataset)}")
if tokenized_dev_dataset:
    print(f"Tokenized development examples: {len(tokenized_dev_dataset)}")

2025-08-28 18:16:33,072 - INFO - Tokenizing datasets...


Map (num_proc=4):   0%|          | 0/482 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenized training examples: 482
Tokenized development examples: 60


## Configure Training Arguments

In [7]:
# Configure training arguments
training_args = TrainingArguments(
    output_dir=str(config.OUTPUT_DIR),
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=config.LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy=config.EVAL_STRATEGY,  # Using the updated config parameter name
    eval_steps=config.EVAL_STEPS,
    save_strategy=config.SAVE_STRATEGY,
    save_steps=config.SAVE_STEPS,
    save_total_limit=config.SAVE_TOTAL_LIMIT,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_dir=str(config.OUTPUT_DIR / "logs"),
    logging_steps=10,
    load_best_model_at_end=True,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),  # Use bf16 if available
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),  # Use fp16 if bf16 not available
    report_to="none",  # Disable wandb reporting by default
    optim="paged_adamw_8bit"  # Use 8-bit AdamW optimizer for memory efficiency
)

# Create data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## Train the Model

In [8]:
def train_model():
    """
    Train the model using the Hugging Face Trainer.
    """
    logger.info("Starting model training...")
    
    # Set up trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_dev_dataset,
        data_collator=data_collator,
    )
    
    # Check for existing checkpoints
    last_checkpoint = None
    if os.path.isdir(training_args.output_dir):
        last_checkpoint = get_last_checkpoint(training_args.output_dir)
        if last_checkpoint is not None:
            logger.info(f"Found checkpoint: {last_checkpoint}")
    
    # Train the model
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    
    # Save the model
    trainer.save_model()
    
    # Save training metrics
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)
    trainer.save_state()
    
    # Save tokenizer
    tokenizer.save_pretrained(training_args.output_dir)
    
    logger.info(f"Training completed. Model saved to {training_args.output_dir}")
    
    return metrics

# Start training
training_metrics = train_model()
print("Training metrics:", training_metrics)

2025-08-28 18:16:34,257 - INFO - Starting model training...
/home/karthikeya/miniconda3/envs/med/lib/python3.9/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


2025-08-28 19:03:58,853 - INFO - Training completed. Model saved to model_output


***** train metrics *****
  epoch                    =        5.0
  total_flos               = 24669677GF
  train_loss               =     0.9892
  train_runtime            = 0:47:23.97
  train_samples_per_second =      0.847
  train_steps_per_second   =      0.107
Training metrics: {'train_runtime': 2843.9753, 'train_samples_per_second': 0.847, 'train_steps_per_second': 0.107, 'total_flos': 2.648886491480064e+16, 'train_loss': 0.9892399303248671, 'epoch': 5.0}


## Define Inference Function

In [4]:
def generate_medical_response(patient_query, model_path=None, max_new_tokens=512, temperature=0.7):
    """
    Generate a medical response for a patient query using the fine-tuned model.
    
    Args:
        patient_query (str): Patient's medical query
        model_path (str, optional): Path to the fine-tuned model. Defaults to training output dir.
        max_new_tokens (int, optional): Maximum number of tokens to generate. Defaults to 512.
        temperature (float, optional): Sampling temperature. Defaults to 0.7.
        
    Returns:
        str: Generated doctor's response
    """
    try:
        # Use default model path if not specified
        if model_path is None:
            model_path = str(config.OUTPUT_DIR)
        
        # Load tokenizer from the specified path
        inference_tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        # Load base model with 4-bit quantization for inference
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
        )
        
        # Load the base model with quantization
        base_model = AutoModelForCausalLM.from_pretrained(
            config.MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        # Load the trained PEFT adapter
        inference_model = PeftModel.from_pretrained(base_model, model_path)
        
        # Format the input text
        input_text = f"Patient: {patient_query}\nDoctor:"
        
        # Tokenize the input
        inputs = inference_tokenizer(input_text, return_tensors="pt").to(inference_model.device)
        
        # Generate the output
        with torch.no_grad():
            outputs = inference_model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                repetition_penalty=1.2,  # Increased to avoid repetitions
                no_repeat_ngram_size=3   # Avoid repeating 3-grams
            )
        
        # Decode the generated output
        generated_text = inference_tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract only the doctor's response
        response_start = generated_text.find("Doctor:")
        if response_start != -1:
            doctor_response = generated_text[response_start + len("Doctor:"):].strip()
        else:
            doctor_response = generated_text[len(input_text):].strip()
        
        # Clean up the response by removing repetitive text patterns
        doctor_response = clean_response(doctor_response)
        
        # Clear CUDA cache to free up memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        return doctor_response
        
    finally:
        # Ensure memory is freed even if an exception occurs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Force garbage collection
        import gc
        gc.collect()

def clean_response(text):
    """
    Clean the response by removing repetitive text and common artifacts.
    
    Args:
        text (str): The raw response text
    
    Returns:
        str: Cleaned response text
    """
    # Remove repetitive "would you like to video or text chat with me" phrases
    if "would you like to video or text chat with me" in text.lower():
        # Find the first occurrence and remove everything after it
        idx = text.lower().find("would you like to video or text chat with me")
        if idx > 0:
            text = text[:idx].strip()
    
    # Remove other common repetitive patterns
    patterns_to_remove = [
        "godspeed",
        "just click the button below",
        "i can answer your questions now",
        "i can review your symptoms",
        "(\\d+/\\d+/\\d+)"  # Date patterns like (3/25/20)
    ]
    
    import re
    for pattern in patterns_to_remove:
        text = re.sub(f"(?i){pattern}", "", text)
    
    # Remove multiple consecutive line breaks
    text = re.sub(r'\n\s*\n', '\n\n', text)
    
    return text.strip()

## Test the Model with Sample Queries

In [5]:
# Sample medical queries for testing
sample_queries = [
    "I've been experiencing persistent headaches for the past week, particularly in the morning. What could be causing this?",
    "My blood pressure readings have been consistently high (around 145/90) despite my medication. Should I be concerned?",
    "I'm diabetic and have noticed some tingling and numbness in my feet. What should I do?",
    "My 5-year-old has a fever of 101°F and is complaining of a sore throat. When should I take them to the doctor?",
    "I've been having trouble sleeping for months. What are some non-medication approaches I could try?"
]

def test_model_with_queries(queries=None, model_path=None, max_queries=None):
    """
    Test the model with a list of sample queries.
    
    Args:
        queries (list, optional): List of queries to test. Defaults to sample_queries.
        model_path (str, optional): Path to the model. Defaults to None.
        max_queries (int, optional): Maximum number of queries to process. Defaults to None.
    """
    if queries is None:
        queries = sample_queries
    
    if max_queries is not None:
        queries = queries[:max_queries]
        
    print("\n=== Testing Fine-tuned Model with Sample Queries ===\n")
    
    for i, query in enumerate(queries, 1):
        print(f"Query {i}: {query}")
        try:
            # Generate response
            response = generate_medical_response(query, model_path=model_path)
            
            # Print a clean, formatted response
            print(f"\nResponse {i}:\n{'-' * 40}")
            print(f"{response}")
            print(f"{'-' * 40}\n")
            
            # Force memory cleanup between queries
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            import gc
            gc.collect()
            
        except Exception as e:
            print(f"Error generating response: {str(e)}\n")
        
        # Add a clear separation between queries
        print("=" * 100)
        print()

# Test the model with sample queries
test_model_with_queries()


=== Testing Fine-tuned Model with Sample Queries ===

Query 1: I've been experiencing persistent headaches for the past week, particularly in the morning. What could be causing this?


2025-08-28 19:55:36,546 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Response 1:
----------------------------------------
in brief: many possibilities. most people get a headache at some time. usually it is just that - a headachr. however there are many causes and some are serious such as meningitis, encephalitis or a brain tumor. these are rare. more common are viruses, colds, flu, sinusitis, allergies, lack of sleep, too much driving, stress, skipping meals, excessive alcohol, sudden changes in weather, long periods on the computer or watching tv.
----------------------------------------


Query 2: My blood pressure readings have been consistently high (around 145/90) despite my medication. Should I be concerned?


...


Response 4:
----------------------------------------
in brief: if concerns yes, definitely call your doctor first to check on current local flu/covid conditions and recommendations. if known covid-19 contact or travel history, or you've been exposed, notify your doctor and arrange for testing. otherwise, monitor temp, which may rise with convulsions if bacterial infection; also watch for signs of worsening cough, shortness of breath, wheezing, or other signs of distress.
----------------------------------------


Query 5: I've been having trouble sleeping for months. What are some non-medication approaches I could try?


2025-08-28 19:57:18,221 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Response 5:
----------------------------------------
many options. here are a few that may help:1. establish regular sleep hours as best you can.2. maintain a comfortable sleep environment.3. create a bedtime routine that is relaxing and calming such as reading, taking a warm bath or meditating.4. limit caffeine, alcohol and heavy meals close to bed time.5. exercise regularly during the day.6. consider using essential oils in a diffuser such as lavender or ylang ylang which are known to promote relaxation.7. try listening to binaural beats on youtube that are designed to put you into delta waves (deep sleep).8. stay away from screens (tv, computer, phone) at least 1 hour before bed time..wish you a good night sleep! let me know if i can assist further.dr jolanda.ps. as a pharmacist i am unable to prescribe medication but can certainly give advice on over the counter sleep aids if needed.
----------------------------------------




## Evaluate Model on Dev Set

In [6]:
def evaluate_model_on_dev(num_samples=5, model_path=None):
    """
    Evaluate the model on a subset of the development set.
    
    Args:
        num_samples (int, optional): Number of samples to evaluate. Defaults to 5.
        model_path (str, optional): Path to the model. Defaults to None.
    """
    if not dev_data:
        print("No development data available for evaluation.")
        return
    
    # Select a random subset of dev samples
    import random
    samples = random.sample(dev_data, min(num_samples, len(dev_data)))
    
    print("\n=== Evaluating Model on Development Set Samples ===\n")
    
    for i, sample in enumerate(samples, 1):
        patient_query = sample.get("patient_query", "")
        actual_response = sample.get("doctor_response", "")
        
        if not patient_query:
            continue
            
        print(f"Sample {i}:")
        print(f"Patient Query: {patient_query}")
        print(f"{'-' * 40}")
        print("Actual Doctor Response:")
        print(actual_response)
        print(f"{'-' * 40}")
        
        try:
            print("\nGenerated Doctor Response:")
            generated_response = generate_medical_response(patient_query, model_path=model_path)
            print(f"{'-' * 40}")
            print(generated_response)
            print(f"{'-' * 40}")
            
            # Force memory cleanup between samples
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            import gc
            gc.collect()
            
        except Exception as e:
            print(f"Error generating response: {str(e)}")
            
        print("\n" + "=" * 100 + "\n")

# Add a function to evaluate specific queries
def evaluate_custom_queries(queries, model_path=None):
    """
    Evaluate the model on custom queries.
    
    Args:
        queries (list): List of query strings to evaluate
        model_path (str, optional): Path to the model. Defaults to None.
    """
    print("\n=== Evaluating Model on Custom Queries ===\n")
    
    for i, query in enumerate(queries, 1):
        print(f"Query {i}: {query}")
        print(f"{'-' * 40}")
        
        try:
            response = generate_medical_response(query, model_path=model_path)
            print("Response:")
            print(f"{'-' * 40}")
            print(response)
            print(f"{'-' * 40}")
            
            # Force memory cleanup between queries
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            import gc
            gc.collect()
            
        except Exception as e:
            print(f"Error generating response: {str(e)}")
            
        print("\n" + "=" * 100 + "\n")

# Evaluate the model on development set samples
evaluate_model_on_dev(num_samples=3)

NameError: name 'dev_data' is not defined

## Save and Load Utilities

In [12]:
def save_training_config():
    """
    Save the training configuration to a JSON file.
    """
    config_dict = {
        "model_name": config.MODEL_NAME,
        "lora_rank": config.LORA_RANK,
        "lora_alpha": config.LORA_ALPHA,
        "lora_dropout": config.LORA_DROPOUT,
        "num_epochs": config.NUM_EPOCHS,
        "learning_rate": config.LEARNING_RATE,
        "batch_size": config.BATCH_SIZE,
        "gradient_accumulation_steps": config.GRADIENT_ACCUMULATION_STEPS,
        "max_sequence_length": config.MAX_SEQUENCE_LENGTH,
        "training_date": str(os.popen('date').read().strip())
    }
    
    config_file = config.OUTPUT_DIR / "training_config.json"
    with open(config_file, 'w', encoding='utf-8') as f:
        json.dump(config_dict, f, indent=2)
        
    logger.info(f"Training configuration saved to {config_file}")

# Save the training configuration
save_training_config()

2025-08-28 19:04:07,659 - INFO - Training configuration saved to model_output/training_config.json


## Create Simple Medical Chatbot Interface

In [ ]:
def medical_chatbot_interface(model_path=None):
    """
    Simple interactive interface for the medical chatbot.
    Type 'quit' or 'exit' to end the conversation.
    
    Args:
        model_path (str, optional): Path to the model. Defaults to None.
    """
    print("\n=== Medical Chatbot Interface ===\n")
    print("Type 'quit' or 'exit' to end the conversation.")
    
    while True:
        user_input = input("\nPatient: ").strip()
        if user_input.lower() in ["quit", "exit"]:
            print("\nThank you for using the Medical Chatbot.")
            break
            
        if not user_input:
            continue
            
        try:
            response = generate_medical_response(user_input, model_path=model_path)
            print(f"\nDoctor: {response}")
            
            # Force memory cleanup after each interaction
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            import gc
            gc.collect()
            
        except Exception as e:
            print(f"\nError: {str(e)}")

# Uncomment the line below to start the interactive chatbot interface
# medical_chatbot_interface()